In [22]:
import numpy as np
import plotly.graph_objects as go
import plotly.express as px

def jednacina(a, b):
    jedn=np.array([a[0]*b[0], a[1]*b[0], a[2]*b[0], a[0]*b[1], a[1]*b[1], a[2]*b[1], a[0]*b[2], a[1]*b[2], a[2]*b[2]])
    return jedn

def fundamentalna(leve, desne):
    MFM=[]
    for i in range(len(leve)):
        jed=jednacina(leve[i], desne[i])
        MFM.append(jed)
    MFM=np.array(MFM)
    _, _, Vh = np.linalg.svd(MFM)
    f=Vh[-1].reshape(3, 3)

    #Popravljanje fundamentalne matrice
    U, S, VhF = np.linalg.svd(f)
    S[2]=0
    Fr = U @ np.diag(S) @ VhF
    return Fr

def osnovna(K, F):
    E= K.T @ F @ K
    Uo, So, Vho = np.linalg.svd(E)
    So = np.diag([1, 1, 0])
    E = Uo @ So @ Vho
    return E

def dekompozicija(E):
    Q0 = np.array([[0,-1,0],
              [1, 0,0],
              [0, 0,1]])

    E0 = np.array([[0, 1,0],
              [-1,0,0],
              [0, 0,0]])

    Ue, Se, Vhe = np.linalg.svd(E)
    if(np.linalg.det(Ue)* np.linalg.det(Vhe)<0):
        E=-E
        Ue, Se, Vhe = np.linalg.svd(E)
    if(np.linalg.det(Ue)<0 and np.linalg.det(Vhe)<0):
        Ue=-Ue
        Vhe=-Vhe
    
    Ec=Ue @ E0 @ Ue.T
    A= Ue @ Q0 @ Vhe
    return [Ec, A]

def vektor(Ec):
    return np.array([Ec[2, 1], Ec[0, 2], Ec[1, 0]])

def matrice_kamera(K, A, t):
    T2 = np.hstack([K, np.zeros((3, 1))])
    t=np.asarray(t).reshape(3, 1)
    T1 = np.hstack([K @ A, K @ t])
    return [T1, T2]

def jed_tr(T1, T2, M1, M2):
    return np.array([M1[1]*T1[2]-M1[2]*T1[1], -M1[0]*T1[2]+M1[2]*T1[0], M2[1]*T2[2]-M2[2]*T2[1], -M2[0]*T2[2]+M2[2]*T2[0]])

def aff(P):
    return P[:-1]/P[-1]

def triangulisi(T1, T2, M1, M2):
    jedM=jed_tr(T1, T2, M1, M2)
    _, _, Vt = np.linalg.svd(jedM)
    M=Vt[-1]
    return aff(M)

def triangulacije_tacaka(leve, desne, T1, T2):
    P3D=[]
    for Ml, Mr in zip(leve, desne):
        Pt=triangulisi(T1, T2, Ml, Mr)
        P3D.append(Pt)
    P3D=np.array(P3D)
    return P3D

def prikazKocke(niz): 
    ivice = np.array([[0,1],[1,2],[2,3],[3,0],[4,5],[5,6],[6,7],[7,4],[0,7],[1,6],[2,5],[3,4],
                  [8,9],[9,10],[10,11],[11,8],[12,13],[13,14],[14,15],[15,12],[8,15],[9,14],[10,13],[11,12]])

    xdata = (np.transpose(niz))[0]
    ydata = (np.transpose(niz))[1]
    zdata = (np.transpose(niz))[2]
    data1 = []
    for i in range(len(ivice)):
        data1.append(go.Scatter3d(x=[xdata[ivice[i][0]], xdata[ivice[i][1]]], y=[ydata[ivice[i][0]], ydata[ivice[i][1]]],z=[zdata[ivice[i][0]], zdata[ivice[i][1]]]))
    fig = go.Figure(data = data1 )
    fig. update_layout(showlegend=False)
    fig.show()

    # fig.write_html("c:/Users/vasfolder/test.html", include_plotlyjs = 'cdn', default_width = '3000px', default_height = '2000px', full_html = False) #Modifiy the html file
    # fig.show()


leve=np.array([1, -1, -1])*(np.array([4032, 0, 0])-np.array([[421, 1151, 1], [1394, 876, 1], [1991, 1385, 1], [942, 1808, 1], [1042, 2583, 1], [1888, 2107, 1], [1404, 1605, 1], [595, 1943, 1], [2526, 818, 1], [3097, 694, 1], [3718, 1086, 1], [3151, 1282, 1], [2850, 1953, 1], [3382, 1720, 1], [2863, 1304, 1], [2353, 1472, 1]]))
desne=np.array([1, -1, -1])*(np.array([4032, 0, 0])-np.array([[1200, 409, 1], [1747, 504, 1], [1235, 792, 1], [646, 645, 1], [831, 1339, 1], [1367, 1516, 1], [1822, 1175, 1], [1313, 1035, 1], [2449, 800, 1], [3413, 1067, 1], [2871, 1657, 1], [1785, 1226, 1], [1859, 2078, 1], [2756, 2583, 1], [3232, 1958, 1], [2410, 1595, 1]]))

F=fundamentalna(leve, desne)
print("Fundamentalna matrica F: \n", F)

K=np.array([3200, 0, 2016, 0, 3200, 1512, 0, 0, 1])
K=K.reshape(3, 3)

E=osnovna(K, F)
print("\nOsnovna matrica E: \n", E)

DE=dekompozicija(E)
Ec=DE[0]
A=DE[1]
print("\nEc matrica: \n", Ec)
print("\nA matrica: \n", A)
# EC_AA = Ec @ A
# print("\nProvera: \n", (E[0, 0] / EC_AA[0, 0]) * EC_AA)

C=vektor(Ec)
C1=-A.T @ C
# print("\nPozicija prve kamere u sistemu druge kamere: \n", C1)

T=matrice_kamera(K, A, C1)
print("\nMatrica prve kamere T1: \n", T[0])
print("\nMatrica druge kamere T2: \n", T[1])

tacke3D=triangulacije_tacaka(leve, desne, T[0], T[1])
print("\n3D koordinate temena:\n", tacke3D)

prikazKocke(tacke3D)




Fundamentalna matrica F: 
 [[ 2.76682468e-07 -1.54431884e-07  1.23815730e-04]
 [ 4.12020939e-07  2.35482868e-07 -3.26022852e-04]
 [-9.32109292e-04 -7.72060180e-04  9.99999207e-01]]

Osnovna matrica E: 
 [[ 0.53043027 -0.51556192  0.23996307]
 [ 0.65589976  0.53529266  0.44898137]
 [ 0.20191569 -0.65968791  0.03118139]]

Ec matrica: 
 [[ 2.45770626e-17 -7.23235532e-01  2.85784884e-01]
 [ 7.23235532e-01  1.03129608e-17  6.28694970e-01]
 [-2.85784884e-01 -6.28694970e-01  7.96847014e-18]]

A matrica: 
 [[-0.10378681 -0.50546699 -0.85658124]
 [ 0.36834459 -0.81952793  0.43897179]
 [-0.923878   -0.26995759  0.27124222]]

Matrica prve kamere T1: 
 [[-2.19465582e+03 -2.16172886e+03 -2.19423567e+03 -1.41541808e+02]
 [-2.18200837e+02 -3.03066525e+03  1.81482797e+03 -9.43214962e+02]
 [-9.23878000e-01 -2.69957590e-01  2.71242217e-01 -8.60151830e-01]]

Matrica druge kamere T2: 
 [[3.200e+03 0.000e+00 2.016e+03 0.000e+00]
 [0.000e+00 3.200e+03 1.512e+03 0.000e+00]
 [0.000e+00 0.000e+00 1.000e+00 0.0